In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
def check_existence(pokedex_no):
    """
    Doc String
    """
    existence_query = f"""
        SELECT 
            *
        FROM 
            {LOOKUPS_DATABASE_PREFIX}.pokedex_updated
        WHERE 
            pokedex_no = {pokedex_no}
    """

    lookup_df = spark.sql(existence_query)

    return not lookup_df.isEmpty()


In [0]:
def api_extraction(url, schema, base_dict = None):
    """
    Doc String
    """
    s_start_time = time.time()

    response = requests.get(url)
    data = response.json()

    s_end_time = time.time()
    time.sleep(s_end_time - s_start_time)

    if not base_dict is None:
        data = base_dict | data

    df = spark.createDataFrame([data], schema=schema)

    return df

In [0]:
def species_extraction(url, schema):
    """
    Doc String
    """
    species_df = api_extraction(url, schema)

    species_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.species")

    return species_df

In [0]:
def varieties_extraction(species_df):
    """
    Doc String
    """
    varieties = species_df.select('varieties').collect()[0][0]

    varieties_list = [[row['is_default'], row['pokemon']['name'], row['pokemon']['url']] for row in varieties]

    for is_default, variety_name, varieties_url in varieties_list:
        base_dict = {
            'base_pokedex_number' : pokedex_no,
            'base_name' : name
        }

        varieties_df = api_extraction(varieties_url, varieties_schema, base_dict)

        varieties_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.varieties")

    return varieties_df

In [0]:
nat_dex_url = f'https://pokeapi.co/api/v2/pokedex/1/'
nat_dex_df = species_extraction(nat_dex_url, nat_dex_schema)

nat_dex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.nat_dex")

pokemon_entries = spark.sql(f"""
    SELECT pokemon_entries FROM staging.nat_dex
""").collect()[0][0]

pokemon_entries_list = [[row['entry_number'], row['pokemon_species']['name'], row['pokemon_species']['url']] for row in pokemon_entries]

for pokedex_no, name, species_url in pokemon_entries_list:
    print(f"{pokedex_no} {name} {species_url}")

    exists = check_existence(pokedex_no)

    if exists:
        print(f"Skipping {pokedex_no} {name} as it has already been extracted")
    else:
        species_df = species_extraction(species_url, species_schema)

        varieties_df = varieties_extraction(species_df)

        insert_query = f"""
            INSERT INTO {LOOKUPS_DATABASE_PREFIX}.pokedex_updated
            VALUES ({pokedex_no}, '{name}', '{datetime.now(timezone.utc).replace(tzinfo = None)}')
        """
        spark.sql(insert_query)